<a href="https://colab.research.google.com/github/Karsuman4298/Generative-AI/blob/main/nanoVLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import torch,math,random
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset,DataLoader
from PIL import Image,ImageDraw
import numpy as np
import matplotlib.pyplot as plt
import numpy as np

In [4]:
IMG_SIZE=32
EMBED_DIM=64
ATTENTION_HEAD=4
BATCH_SIZE=12
EPOCH=10
LR=3e-4
TEMPERATURE=0.08
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')




# SYNTHETIC DATASET

In [6]:
color=['red','green','blue','yellow','purple','orange','pink','brown','grey']
shapes=['square','circle','rectangle']
positions=['left','center','right','top','bottom','top-left','top-right','bottom-left','bottom-right']

# DRAWING IMAGE SHAPES

In [9]:
def draw_sample(color, position, shape, img_size=IMG_SIZE):
  img=Image.new('RGB',(img_size,img_size),'white')
  draw=ImageDraw.Draw(img)
  margin=6
  h=w=img_size-2*margin

  # Calculate x coordinates
  if 'left' in position:
      x0 = margin
      x1 = margin + w // 2
  elif 'top-left' in position:
      x0 = margin
      x1 = margin + w // 2
  elif 'bottom-left' in position:
      x0 = margin
      x1 = margin + w // 2
  elif 'right' in position:
      x0 = margin + w // 2
      x1 = img_size - margin
  elif 'top-right' in position:
      x0 = margin + w // 2
      x1 = img_size - margin
  elif 'bottom-right' in position:
      x0 = margin + w // 2
      x1 = img_size - margin
  else: # center or vertical positions
      x0 = margin + w // 4
      x1 = margin + 3 * w // 4

  # Calculate y coordinates
  if 'top' in position:
      y0 = margin
      y1 = margin + h // 2
  elif 'top-left' in position:
      y0 = margin
      y1 = margin + h // 2
  elif 'top-right' in position:
      y0 = margin
      y1 = margin + h // 2
  elif 'bottom' in position:
      y0 = margin + h // 2
      y1 = img_size - margin
  elif 'bottom-left' in position:
      y0 = margin + h // 2
      y1 = img_size - margin
  elif 'bottom-right' in position:
      y0 = margin + h // 2
      y1 = img_size - margin
  else: # center or horizontal positions
      y0 = margin + h // 4
      y1 = margin + 3 * h // 4

  if shape == 'square':
      draw.rectangle([x0, y0, x1, y1], fill=color, outline='black')
  elif shape == 'circle':
      draw.ellipse([x0, y0, x1, y1], fill=color, outline='black')
  else: # triangle
      draw.polygon([(x0 + (x1 - x0) // 2, y0), (x0, y1), (x1, y1)], fill=color, outline='black')
  return img

In [10]:
class ShapeDataset:
  def __init__(self):
   self.image= []
   self.captions=[]

   for c in color:
    for p in positions:
      for s in shapes:
        img=draw_sample(s,p,c)
        cap= f"{c} {s} {p}"
        self.image.append(torch.from_numpy(np.asarray(img)).permute(2,0,1).float()//255.0)
        self.captions.append(cap)


  def build_vocab(self,texts):

